# Phase 2: # Phase 2: Feature Relevance for Demand Forecasting & Safety Stock

This section reviews all columns for their relevance to demand forecasting and safety stock calculation. It documents which features are included or excluded, and the rationale for each decision.

## 1. Import Required Libraries
Import pandas, numpy, matplotlib, seaborn, and any other libraries needed for data analysis and visualization.

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set(style="whitegrid")

## 2. Load Dataset

In [2]:

df = pd.read_csv('/home/marc/project/supply_chain_DS/kaggle_supply_chain_dataSet/supply_chain_data.csv')
df.head()

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632


In [7]:
df.shape

(100, 24)

## 3. Feature Relevance Table

The following table summarizes the relevance of each column for demand forecasting and safety stock calculation. Columns are marked as 'Include' or 'Exclude' with rationale.

In [5]:
# Define relevance for each column
relevance = {
    'Product Type': ('Include', 'Product category, useful for grouping and seasonality'),
    'SKU': ('Include', 'Unique identifier for each product, essential for SKU-level analysis'),
    'Price': ('Include', 'May influence demand'),
    'Availability': ('Include', 'Directly impacts ability to fulfill demand'),
    'Number of products sold': ('Include', 'Target variable for demand forecasting'),
    'Revenue generated': ('Include', 'Correlated with demand, useful for business KPIs'),
    'Customer demographics': ('Exclude', 'Not directly relevant unless granular and available per order'),
    'Stock levels': ('Include', 'Key for safety stock calculation'),
    'Lead times': ('Include', 'Critical for safety stock and replenishment'),
    'Order quantities': ('Include', 'Related to demand and inventory flow'),
    'Shipping times': ('Include', 'Affects lead time and service level'),
    'Shipping carriers': ('Include', 'May impact shipping reliability and lead time'),
    'Shipping costs': ('Include', 'May affect order size and profitability'),
    'Supplier name': ('Include', 'Supplier reliability can affect lead time and stockouts'),
    'Location': ('Include', 'May affect demand and logistics'),
    'Production volumes': ('Include', 'Affects supply capability'),
    'Manufacturing lead time': ('Include', 'Impacts replenishment and safety stock'),
    'Manufacturing costs': ('Include', 'Relevant for profitability, less so for demand'),
    'Inspection results': ('Include', 'Quality issues can affect available stock'),
    'Defect rates': ('Include', 'Affects usable inventory'),
    'Transportation modes': ('Include', 'Impacts lead time and cost'),
    'Routes': ('Include', 'May affect lead time and reliability'),
    'Costs': ('Include', 'General cost information, useful for business KPIs'),
}

relevance_summary = pd.DataFrame([
    {
        'Column': col,
        'Relevance': relevance.get(col, ('Review', 'Needs further assessment'))[0],
        'Rationale': relevance.get(col, ('Review', 'Needs further assessment'))[1]
    }
    for col in df.columns
])
display(relevance_summary)

,Column,Relevance,Rationale
0,Product type,Review,Needs further assessment
1,SKU,Include,"Unique identifier for each product, essential ..."
2,Price,Include,May influence demand
3,Availability,Include,Directly impacts ability to fulfill demand
4,Number of products sold,Include,Target variable for demand forecasting
5,Revenue generated,Include,"Correlated with demand, useful for business KPIs"
6,Customer demographics,Exclude,Not directly relevant unless granular and avai...
7,Stock levels,Include,Key for safety stock calculation
8,Lead times,Include,Critical for safety stock and replenishment
9,Order quantities,Include,Related to demand and inventory flow


---

**Summary:**
- All relevant columns for demand forecasting and safety stock have been identified.
- Irrelevant columns (e.g., Customer demographics) are excluded unless further granularity is available.
- The rationale for each inclusion/exclusion is documented in the table above.

Proceed to feature engineering based on the selected columns.

## 4. Feature Engineering


### Proposed Engineered Features

The following features are engineered to enhance demand forecasting and safety stock analysis:

- **Sales Velocity:** Products sold per lead time unit.
- **Lead Time Variability:** Absolute difference between 'Lead times' and 'Manufacturing lead time'.
- **Stockout Risk:** Ratio of stock levels to order quantities.
- **Revenue per Unit Sold:** Revenue generated divided by number of products sold.
- **Defect Rate Category:** Binned defect rates (Low/Medium/High).
- **Supplier Reliability:** Binary indicator based on inspection results and defect rates.

These features are for exploration and methodology prototyping, given the small dataset size.

In [8]:
# Feature Engineering Implementation

# 1. Sales velocity (products sold per lead time unit)
df['sales_velocity'] = df['Number of products sold'] / df['Lead times']

# 2. Lead time variability (if both columns exist)
if 'Lead times' in df.columns and 'Manufacturing lead time' in df.columns:
    df['lead_time_variability'] = abs(df['Lead times'] - df['Manufacturing lead time'])

# 3. Stockout risk (stock levels to order quantities)
df['stockout_risk'] = df['Stock levels'] / df['Order quantities']

# 4. Revenue per unit sold
df['revenue_per_unit'] = df['Revenue generated'] / df['Number of products sold']

# 5. Defect rate category
df['defect_rate_category'] = pd.cut(df['Defect rates'], bins=[-float('inf'), 1, 3, float('inf')], labels=['Low', 'Medium', 'High'])

# 6. Supplier reliability (example: 1 if inspection pending and defect rate low, else 0)
df['supplier_reliable'] = ((df['Inspection results'] == 'Pending') & (df['Defect rates'] < 1)).astype(int)

df.head()

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Defect rates,Transportation modes,Routes,Costs,sales_velocity,lead_time_variability,stockout_risk,revenue_per_unit,defect_rate_category,supplier_reliable
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,0.226410,Road,Route B,187.752075,114.571429,22,0.604167,10.800495,Low,1
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,4.854068,Road,Route B,503.065579,24.533333,0,1.432432,10.137092,High,0
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,4.580593,Air,Route C,141.920282,0.800000,17,0.011364,1197.218703,High,0
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,4.746649,Rail,Route A,254.776159,6.384615,5,0.389831,93.576342,High,0
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,3.145580,Air,Route A,923.440632,290.333333,0,0.089286,3.084392,High,0


## 5. Business questions

Here are relevant business questions that can be explored using the newly engineered features:

1. Which SKUs have the highest and lowest sales velocity?
→ Helps identify fast- and slow-moving products for inventory optimization.

2. Are there SKUs or suppliers with high lead time variability?
→ Pinpoints supply chain instability and potential risk areas.

3. Which products are at greatest risk of stockouts?
→ Use the stockout risk feature to prioritize replenishment and safety stock.

3. How does revenue per unit sold vary by product type or supplier?
→ Identifies the most profitable products and partners.

4. Are there patterns between defect rate categories and supplier reliability?
→ Supports supplier evaluation and quality improvement initiatives.

5. Do reliable suppliers (as defined) correlate with lower stockout risk or higher sales velocity?
→ Assesses the impact of supplier quality on supply chain performance.

6. How do engineered features (e.g., sales velocity, stockout risk) change over time or by location?
→ Reveals trends and regional differences for targeted actions.

7. Can we segment products or suppliers based on these features for differentiated inventory policies?
→ Enables tailored strategies for different product/supplier groups.

These questions help drive actionable insights for demand planning, inventory management, and supplier performance improvement.

In [9]:
# Business Question 1: Which SKUs have the highest and lowest sales velocity?

# Top 5 SKUs by sales velocity
print("Top 5 SKUs by sales velocity:")
display(df[['SKU', 'sales_velocity']].sort_values(by='sales_velocity', ascending=False).head())

# Bottom 5 SKUs by sales velocity
print("Bottom 5 SKUs by sales velocity:")
display(df[['SKU', 'sales_velocity']].sort_values(by='sales_velocity', ascending=True).head())

Top 5 SKUs by sales velocity:


,SKU,sales_velocity
98,SKU98,913.0
22,SKU22,884.0
38,SKU38,705.0
69,SKU69,511.0
71,SKU71,318.5


Bottom 5 SKUs by sales velocity:


,SKU,sales_velocity
2,SKU2,0.800000
85,SKU85,1.388889
48,SKU48,1.812500
70,SKU70,2.666667
97,SKU97,3.263158


In [10]:
# Business Question 1: Which SKUs have the highest and lowest sales velocity?

# Top 5 SKUs by sales velocity
print("Top 5 SKUs by sales velocity:")
display(df.sort_values(by='sales_velocity', ascending=False).head())

# Bottom 5 SKUs by sales velocity
print("Bottom 5 SKUs by sales velocity:")
display(df.sort_values(by='sales_velocity', ascending=True).head())

Top 5 SKUs by sales velocity:


,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Defect rates,Transportation modes,Routes,Costs,sales_velocity,lead_time_variability,stockout_risk,revenue_per_unit,defect_rate_category,supplier_reliable
98,skincare,SKU98,19.754605,43,913,8525.952560,Female,53,1,27,...,2.908122,Rail,Route A,882.198864,913.0,8,1.962963,9.338393,Medium,0
22,haircare,SKU22,27.679781,55,884,2390.807867,Unknown,71,1,63,...,2.591275,Rail,Route C,205.571996,884.0,27,1.126984,2.704534,Medium,0
38,cosmetics,SKU38,52.075931,75,705,9692.318040,Non-binary,69,1,88,...,0.613327,Air,Route B,339.672870,705.0,11,0.784091,13.747969,Low,1
69,skincare,SKU69,54.865529,62,511,1752.381087,Non-binary,95,1,27,...,1.362388,Air,Route A,207.663206,511.0,6,3.518519,3.429317,Medium,0
71,cosmetics,SKU71,6.381533,14,637,8180.337085,Female,76,2,26,...,2.078751,Road,Route A,405.167068,318.5,8,2.923077,12.841973,Medium,0


Bottom 5 SKUs by sales velocity:


,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Defect rates,Transportation modes,Routes,Costs,sales_velocity,lead_time_variability,stockout_risk,revenue_per_unit,defect_rate_category,supplier_reliable
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,4.580593,Air,Route C,141.920282,0.800000,17,0.011364,1197.218703,High,0
85,cosmetics,SKU85,76.962994,83,25,8684.613059,Female,15,18,66,...,1.374429,Road,Route B,842.686830,1.388889,16,0.227273,347.384522,Medium,0
48,haircare,SKU48,76.035544,28,29,7397.071005,Non-binary,30,16,9,...,1.698113,Rail,Route B,768.651914,1.812500,2,3.333333,255.071414,Medium,0
70,haircare,SKU70,47.914542,90,32,7014.887987,Female,10,12,22,...,1.830576,Road,Route C,183.272899,2.666667,4,0.454545,219.215250,Medium,0
97,haircare,SKU97,3.526111,56,62,4370.916580,Male,46,19,4,...,3.376238,Road,Route A,540.132423,3.263158,6,11.500000,70.498655,High,0


In [12]:
# Business Question 2: Are there SKUs or suppliers with high lead time variability?

# Top 5 SKUs by lead time variability
print("Top 5 SKUs by lead time variability:")
display(df[['SKU', 'lead_time_variability']].sort_values(by='lead_time_variability', ascending=False).head())

# Bottom 5 SKUs by lead time variability
print("Bottom 5 SKUs by lead time variability:")
display(df[['SKU', 'lead_time_variability']].sort_values(by='lead_time_variability', ascending=True).head())

Top 5 SKUs by lead time variability:


,SKU,lead_time_variability
22,SKU22,27
12,SKU12,27
96,SKU96,26
35,SKU35,25
51,SKU51,24


Bottom 5 SKUs by lead time variability:


,SKU,lead_time_variability
1,SKU1,0
4,SKU4,0
15,SKU15,1
17,SKU17,1
44,SKU44,1


3. Which products are at greatest risk of stockouts?
→ Use the stockout risk feature to prioritize replenishment and safety stock.

In [13]:
# Business Question 3: Which products are at greatest risk of stockouts?

# Top 5 SKUs by stockout risk
print("Top 5 SKUs by stockout risk:")
display(df[['SKU', 'stockout_risk']].sort_values(by='stockout_risk', ascending=False).head())

# Bottom 5 SKUs by stockout risk
print("Bottom 5 SKUs by stockout risk:")
display(df[['SKU', 'stockout_risk']].sort_values(by='stockout_risk', ascending=True).head())

Top 5 SKUs by stockout risk:


,SKU,stockout_risk
74,SKU74,41.000000
46,SKU46,15.333333
97,SKU97,11.500000
49,SKU49,10.777778
21,SKU21,9.857143


Bottom 5 SKUs by stockout risk:


,SKU,stockout_risk
68,SKU68,0.000000
2,SKU2,0.011364
16,SKU16,0.025641
33,SKU33,0.042105
34,SKU34,0.047619
